# Notebook 01 — Project Definition & Data Understanding
## Section 1: Project Scope and Shared Configuration

**Why this section**  
We lock the project definition so every later notebook uses the same scope, target, metric, and months.

**Problem**  
Predict ICE stop delay in **minutes** (`delay_in_min`) using railway operations + historical weather.

**Decisions (post mid-presentation)**  
- **Task:** regression only (not classification)  
- **Metric:** MAE only  
- **Months:** 2024-07, 2024-08, 2024-09  
- **Unit of analysis:** one row = one ICE train stop at one station  

**What this section produces**  
`data/reference/project_config.json` — shared config for Notebooks 02–10.

**Expected output**  
Printed summary of authors, target, metric, months, and the saved config path.

In [1]:
# =============================================================================
# Notebook 01 | Section 1: Project Scope and Shared Configuration
# =============================================================================
# Self-contained: creates folders and writes project_config.json
# Aligned with mid-presentation feedback: regression-only, MAE, multi-month
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path


def find_project_root() -> Path:
    """Walk up from cwd until data/reference can be created/found under Notebooks."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        # Prefer Notebooks/ as project root (where data/ lives)
        if (candidate / "data").exists() or candidate.name.lower() == "notebooks":
            if candidate.name.lower() == "notebooks" or (candidate / "data" / "reference").exists():
                return candidate
        # Also accept ML Project root that contains Notebooks/
        nb = candidate / "Notebooks"
        if (nb / "data").exists() or nb.exists():
            return nb
    # Fallback: cwd
    return Path.cwd()


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REFERENCE_DIR = DATA_DIR / "reference"
FIGURES_DIR = PROCESSED_DIR / "figures"

for d in [PROCESSED_DIR, REFERENCE_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = REFERENCE_DIR / "project_config.json"

# --- Project decisions (do not change casually) ---
AUTHORS = ["Manikanta Engalligi", "abhinay-sambherao"]
TARGET_MONTHS = ["2024-07", "2024-08", "2024-09"]
PRIMARY_TARGET = "delay_in_min"
PRIMARY_METRIC = "mae"
TRAIN_TYPE_FILTER = "ICE"

# Verified Deutsche Bahn schema (from real sample inspection — do not invent columns)
DB_VERIFIED_COLUMNS = [
    {"name": "station_name", "dtype": "string", "role": "station label (nullable)"},
    {"name": "xml_station_name", "dtype": "string", "role": "API/XML station name"},
    {"name": "eva", "dtype": "string", "role": "EVA station ID — geocoding key"},
    {"name": "train_number", "dtype": "string", "role": "train service number"},
    {"name": "line_number", "dtype": "string", "role": "line ID (nullable)"},
    {"name": "final_destination_station", "dtype": "string", "role": "route endpoint"},
    {"name": "delay_in_min", "dtype": "int32", "role": "regression target"},
    {"name": "time", "dtype": "timestamp", "role": "record reference time"},
    {"name": "is_canceled", "dtype": "bool", "role": "cancellation flag"},
    {"name": "train_type", "dtype": "string", "role": "filter: ICE only"},
    {"name": "train_line_ride_id", "dtype": "string", "role": "unique ride ID"},
    {"name": "train_line_station_num", "dtype": "int32", "role": "stop order on route"},
    {"name": "arrival_planned_time", "dtype": "timestamp", "role": "scheduled arrival"},
    {"name": "arrival_change_time", "dtype": "timestamp", "role": "actual/revised arrival"},
    {"name": "departure_planned_time", "dtype": "timestamp", "role": "scheduled departure"},
    {"name": "departure_change_time", "dtype": "timestamp", "role": "actual/revised departure"},
    {"name": "id", "dtype": "string", "role": "unique row ID"},
]

project_config = {
    "project": {
        "title": "ICE Train Delay Prediction Using Railway Operational Data and Historical Weather Data",
        "authors": AUTHORS,
        "institution": "Your University",
        "notebook": "01_Project_Definition_and_Data_Understanding",
        "section": "Section 1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "revision_note": "Aligned with mid-presentation feedback: regression-only, MAE primary, multi-month scope",
    },
    "scope": {
        "train_type_filter": TRAIN_TYPE_FILTER,
        "observation_unit": "one train stop at one station (one row)",
        "target_months": TARGET_MONTHS,
        "exclude_from_git": "full raw Deutsche Bahn dataset (54.8 GB)",
    },
    "ml_tasks": {
        "primary": {
            "task_type": "regression",
            "question": "How many minutes late will this ICE train stop be?",
            "target": PRIMARY_TARGET,
            "target_source": "column in Deutsche Bahn dataset",
            "primary_metric": PRIMARY_METRIC,
            "metric_rationale": (
                "MAE is the average absolute error in minutes — easy to interpret "
                "and sufficient as the single optimization metric."
            ),
        },
        "not_in_scope": {
            "classification": (
                "Binary delayed/not-delayed modeling removed after mid-presentation "
                "feedback; delay magnitude is the primary objective."
            ),
        },
    },
    "data_sources": {
        "deutsche_bahn": {
            "hf_dataset_id": "piebro/deutsche-bahn-data",
            "hf_url": "https://huggingface.co/datasets/piebro/deutsche-bahn-data",
            "monthly_parquet_pattern": "monthly_processed_data/data-{year_month}.parquet",
            "approx_rows": 187311724,
            "approx_size_gb": 54.8,
            "license": "CC BY 4.0 (Deutsche Bahn attribution required)",
        },
        "open_meteo": {
            "archive_url": "https://archive-api.open-meteo.com/v1/archive",
            "docs_url": "https://open-meteo.com/en/docs/historical-weather-api",
            "required_params": ["latitude", "longitude", "start_date", "end_date", "hourly"],
            "candidate_hourly_variables": [
                "temperature_2m",
                "precipitation",
                "rain",
                "snowfall",
                "windspeed_10m",
                "windgusts_10m",
                "weather_code",
                "visibility",
            ],
        },
    },
    "db_verified_columns": DB_VERIFIED_COLUMNS,
    "db_missing_columns": [
        {"name": "latitude", "solution": "geocode eva/xml_station_name in Notebook 04"},
        {"name": "longitude", "solution": "geocode eva/xml_station_name in Notebook 04"},
    ],
    "paths": {
        "project_root": str(PROJECT_ROOT),
        "data_dir": str(DATA_DIR),
        "processed_dir": str(PROCESSED_DIR),
        "reference_dir": str(REFERENCE_DIR),
        "config_path": str(CONFIG_PATH),
    },
    "notebook_roadmap": [
        "01 — Project Definition & Data Understanding",
        "02 — Data Acquisition & Initial Inspection (multi-month)",
        "03 — Data Cleaning & Standardization (multi-month)",
        "04 — Data Integration: Merge Deutsche Bahn + Weather (multi-month)",
        "05 — Exploratory Data Analysis (delay minutes + weather)",
        "06 — Feature Engineering",
        "07 — Baseline Regression Models (MAE)",
        "08 — Advanced Regression Models",
        "09 — Model Evaluation, Tuning & Weather Ablation",
        "10 — Final Prediction Pipeline & Conclusion",
    ],
    "reproducibility": {
        "rule": "Each notebook is self-contained; load shared config and data from disk.",
        "config_file": str(CONFIG_PATH),
        "merged_data_pattern": "data/processed/ice_weather_merged_YYYY-MM.parquet",
    },
}

with CONFIG_PATH.open("w", encoding="utf-8") as f:
    json.dump(project_config, f, indent=2, ensure_ascii=False)

# --- Summary ---
sep = "=" * 72
print(sep)
print("Section 1: Project Scope and Shared Configuration")
print(sep)
print(f"Authors       : {', '.join(AUTHORS)}")
print(f"Task          : regression")
print(f"Target        : {PRIMARY_TARGET}")
print(f"Primary metric: {PRIMARY_METRIC.upper()} (MAE only — RMSE not used)")
print(f"Train filter  : {TRAIN_TYPE_FILTER}")
print(f"Target months : {', '.join(TARGET_MONTHS)}")
print(f"Config saved  : {CONFIG_PATH}")
print(f"DB columns    : {len(DB_VERIFIED_COLUMNS)} verified")
print()
print("Not in scope  : classification as a primary ML task")
print(sep)
print("Next: Section 2 — research questions + MAE evaluation framework")
print(sep)

Section 1: Project Scope and Shared Configuration
Authors       : Manikanta Engalligi, abhinay-sambherao
Task          : regression
Target        : delay_in_min
Primary metric: MAE (MAE only — RMSE not used)
Train filter  : ICE
Target months : 2024-07, 2024-08, 2024-09
Config saved  : c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\project_config.json
DB columns    : 17 verified

Not in scope  : classification as a primary ML task
Next: Section 2 — research questions + MAE evaluation framework


# Section 2: Research Questions and Evaluation Framework

**Why this section**  
We define what we will answer and how we will score models, so Notebooks 06–10 stay consistent.

**Primary task**  
Regression: predict `delay_in_min` (minutes late/early).

**Primary metric**  
**MAE only** (Mean Absolute Error in minutes).  
RMSE is not used. Classification is not a primary modeling goal.

**Research questions**  
- **RQ1:** Can operational features alone predict delay in minutes?  
- **RQ2:** Does weather improve MAE vs operational-only models? (ablation)  
- **RQ3:** Which weather variables relate most to delay magnitude?

**What this section produces**  
`data/reference/research_framework.json`

**Expected output**  
Printed RQs, target, MAE rule, months, and saved file path.

In [2]:
# =============================================================================
# Notebook 01 | Section 2: Research Questions and Evaluation Framework
# =============================================================================
# Self-contained: loads project_config.json, writes research_framework.json
# Regression-only | MAE only | months 2024-07..2024-09
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError(
        "Cannot find data/reference/project_config.json. Run Section 1 first."
    )


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"
FRAMEWORK_PATH = REFERENCE_DIR / "research_framework.json"

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    project_config = json.load(f)

TARGET_MONTHS = project_config["scope"]["target_months"]
PRIMARY_TARGET = project_config["ml_tasks"]["primary"]["target"]
PRIMARY_METRIC = project_config["ml_tasks"]["primary"]["primary_metric"]

research_framework = {
    "metadata": {
        "notebook": "01_Project_Definition_and_Data_Understanding",
        "section": "Section 2",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "depends_on": str(CONFIG_PATH),
        "revision_note": (
            "Regression-only; MAE as sole primary metric; "
            "multi-month scope (2024-07 to 2024-09)"
        ),
    },
    "research_questions": {
        "RQ1": {
            "question": "Can operational features alone predict ICE stop delay in minutes?",
            "answered_in": ["Notebook 07", "Notebook 09"],
            "method": "Train regression baselines using only railway features; evaluate with MAE.",
        },
        "RQ2": {
            "question": "Does adding historical weather data improve delay (minutes) prediction?",
            "answered_in": ["Notebook 07", "Notebook 09"],
            "method": "Weather ablation: same regressor with vs without weather features; compare MAE.",
        },
        "RQ3": {
            "question": "Which weather variables are most associated with ICE delay magnitude?",
            "answered_in": ["Notebook 05", "Notebook 09"],
            "method": "EDA correlations/bin analysis + feature importance in final regressors.",
        },
    },
    "regression": {
        "task_type": "regression",
        "target_column": PRIMARY_TARGET,
        "target_filter": (
            "Non-canceled stops; delay can be negative (early) or positive (late)."
        ),
        "primary_metric": PRIMARY_METRIC,
        "secondary_metrics": [],
        "metric_rationale": (
            "MAE (Mean Absolute Error) is the average absolute error in minutes. "
            "It is interpretable and is the single metric we optimize and report."
        ),
        "naive_baselines": {
            "mean_predictor": "always predict mean delay_in_min on the training period",
            "median_predictor": "always predict median delay_in_min on the training period",
        },
    },
    "not_in_scope": {
        "classification": {
            "reason": (
                "Removed as a primary ML task after mid-presentation feedback. "
                "Predicting delay magnitude (minutes) is sufficient and clearer."
            ),
            "note": (
                "EDA may still describe the delay distribution; "
                "no binary classification models will be trained."
            ),
        },
        "rmse": {
            "reason": "Not used as a primary or required metric; MAE alone is the evaluation criterion.",
        },
    },
    "hypotheses": {
        "H1": {
            "statement": (
                "Higher hourly precipitation is associated with higher expected delay (minutes)."
            ),
            "type": "regression",
            "test": (
                "Compare mean/median delay across precipitation bins in EDA; "
                "check precipitation importance in regressors."
            ),
        },
        "H2": {
            "statement": "Higher wind gusts increase expected delay magnitude.",
            "type": "regression",
            "test": "Correlation/EDA vs delay_in_min; wind feature importance in final models.",
        },
        "H3": {
            "statement": (
                "Models with weather features achieve lower MAE than operational-only models."
            ),
            "type": "regression",
            "test": (
                "Ablation study: same algorithm, with vs without weather columns; compare MAE."
            ),
        },
    },
    "evaluation_protocol": {
        "train_test_split": "time-based (no random shuffle) — implemented in Notebook 07",
        "primary_metric": PRIMARY_METRIC,
        "baseline_requirement": "Must beat naive mean/median MAE before using advanced models",
        "weather_ablation": {
            "description": "Train identical regressor twice: with and without weather features",
            "purpose": "Directly answers RQ2 using MAE",
        },
    },
    "alternatives_considered": {
        "classification_as_primary_task": (
            "Rejected after mid-presentation — delay minutes are the natural continuous target"
        ),
        "rmse_as_co_primary_metric": (
            "Rejected — MAE alone is sufficient and clearer for evaluation"
        ),
        "single_month_only": (
            "Expanded to July–September 2024 for stronger, more general results"
        ),
        "random_train_test_split": "Rejected — causes temporal data leakage",
    },
    "target_months": TARGET_MONTHS,
}

with FRAMEWORK_PATH.open("w", encoding="utf-8") as f:
    json.dump(research_framework, f, indent=2, ensure_ascii=False)

# --- Summary ---
sep = "=" * 72
print(sep)
print("Section 2: Research Questions and Evaluation Framework")
print(sep)
print("Research questions:")
for key, rq in research_framework["research_questions"].items():
    print(f"  {key}: {rq['question']}")
print()
print(f"Target           : {PRIMARY_TARGET}")
print(f"Primary metric   : {PRIMARY_METRIC.upper()} only")
print(f"Not used         : RMSE, classification models")
print(f"Target months    : {', '.join(TARGET_MONTHS)}")
print(f"Framework saved  : {FRAMEWORK_PATH}")
print(sep)
print("Next: Section 3 — inspect Deutsche Bahn schema (real columns only)")
print(sep)

Section 2: Research Questions and Evaluation Framework
Research questions:
  RQ1: Can operational features alone predict ICE stop delay in minutes?
  RQ2: Does adding historical weather data improve delay (minutes) prediction?
  RQ3: Which weather variables are most associated with ICE delay magnitude?

Target           : delay_in_min
Primary metric   : MAE only
Not used         : RMSE, classification models
Target months    : 2024-07, 2024-08, 2024-09
Framework saved  : c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\research_framework.json
Next: Section 3 — inspect Deutsche Bahn schema (real columns only)


# Section 3: Deutsche Bahn Schema & ICE Scope

**Why this section**  
We inspect **real** Deutsche Bahn columns before any modeling. We never invent fields.

**Approach**  
Fetch a **small sample (100 rows)** via Hugging Face Dataset Viewer API — not the full ~54.8 GB dataset.

**What we check**  
- Confirmed 17 columns  
- `train_type` distribution (sample includes many types; we keep **ICE only** later)  
- `delay_in_min` = **regression target**  
- Missing lat/lon → geocode in Notebook 04  

**Months for this project**  
2024-07, 2024-08, 2024-09 (full monthly loads start in Notebook 02).

**Output**  
`data/reference/db_data_dictionary.json`

In [3]:
# =============================================================================
# Notebook 01 | Section 3: Deutsche Bahn Data Dictionary & ICE Scope
# =============================================================================
# Self-contained: loads project_config.json; fetches small HF sample via API.
# Saves db_data_dictionary.json for Notebooks 02–04.
# Does NOT download the full 54.8 GB dataset.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import requests


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError(
        "Cannot find data/reference/project_config.json. Run Section 1 first."
    )


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"
DB_DICTIONARY_PATH = REFERENCE_DIR / "db_data_dictionary.json"

HF_DATASET_ID = "piebro/deutsche-bahn-data"
HF_ROWS_API = "https://datasets-server.huggingface.co/rows"
SAMPLE_LENGTH = 100
REQUEST_TIMEOUT_SEC = 60
ICE_FILTER = "ICE"


def load_project_config(path: Path = CONFIG_PATH) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}\nRun Notebook 01 Section 1 first.")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_db_data_dictionary(path: Path = DB_DICTIONARY_PATH) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}\nRun Notebook 01 Section 3 first.")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


config = load_project_config()
VERIFIED_COLUMNS = [c["name"] for c in config["db_verified_columns"]]
TARGET_MONTHS = config["scope"]["target_months"]
PRIMARY_TARGET = config["ml_tasks"]["primary"]["target"]

# =============================================================================
# Column dictionary (verified schema — Hugging Face Dataset Viewer)
# =============================================================================
DB_DATA_DICTIONARY: dict[str, dict[str, Any]] = {
    "station_name": {
        "dtype": "string",
        "nullable": True,
        "description": "Human-readable station name. Can be null; prefer xml_station_name or eva.",
        "ml_role": "feature",
        "merge_key": False,
    },
    "xml_station_name": {
        "dtype": "string",
        "nullable": False,
        "description": "Station name as returned by Deutsche Bahn XML/API.",
        "ml_role": "feature",
        "merge_key": False,
    },
    "eva": {
        "dtype": "string",
        "nullable": False,
        "description": "EVA number — official German station identifier.",
        "ml_role": "key",
        "merge_key": True,
        "note": "Primary station key for geocoding before weather merge.",
    },
    "train_number": {
        "dtype": "string",
        "nullable": False,
        "description": "Train service number.",
        "ml_role": "feature",
        "merge_key": False,
    },
    "line_number": {
        "dtype": "string",
        "nullable": True,
        "description": "Line identifier. Often null for ICE long-distance services.",
        "ml_role": "feature",
        "merge_key": False,
    },
    "final_destination_station": {
        "dtype": "string",
        "nullable": False,
        "description": "Terminal station of the train route.",
        "ml_role": "feature",
        "merge_key": False,
    },
    "delay_in_min": {
        "dtype": "int32",
        "nullable": False,
        "description": "Delay in minutes. Negative = early; 0 = on time; positive = late.",
        "ml_role": "target",
        "merge_key": False,
        "note": "Primary regression target for this project. Evaluated with MAE.",
    },
    "time": {
        "dtype": "timestamp[ns]",
        "nullable": False,
        "description": "Reference timestamp for the record.",
        "ml_role": "feature",
        "merge_key": False,
        "note": "Standardize to Europe/Berlin in Notebook 03.",
    },
    "is_canceled": {
        "dtype": "bool",
        "nullable": False,
        "description": "True if this train stop was canceled.",
        "ml_role": "filter",
        "merge_key": False,
        "note": "Excluded from modeling in Notebook 03 (no valid delay outcome).",
    },
    "train_type": {
        "dtype": "string",
        "nullable": False,
        "description": "Vehicle category (ICE, IC, RE, S, Bus, etc.).",
        "ml_role": "filter",
        "merge_key": False,
        "note": f"Keep only rows where train_type == '{ICE_FILTER}'.",
    },
    "train_line_ride_id": {
        "dtype": "string",
        "nullable": False,
        "description": "Unique identifier for one complete train journey.",
        "ml_role": "identifier",
        "merge_key": False,
    },
    "train_line_station_num": {
        "dtype": "int32",
        "nullable": False,
        "description": "Stop sequence number on the route (1 = first stop).",
        "ml_role": "feature",
        "merge_key": False,
    },
    "arrival_planned_time": {
        "dtype": "timestamp[ns]",
        "nullable": True,
        "description": "Scheduled arrival at this station.",
        "ml_role": "feature",
        "merge_key": False,
    },
    "arrival_change_time": {
        "dtype": "timestamp[ns]",
        "nullable": True,
        "description": "Actual or revised arrival time.",
        "ml_role": "feature",
        "merge_key": False,
    },
    "departure_planned_time": {
        "dtype": "timestamp[ns]",
        "nullable": True,
        "description": "Scheduled departure from this station.",
        "ml_role": "feature",
        "merge_key": True,
        "note": "Primary temporal anchor for hour-level weather merge (Notebook 04).",
    },
    "departure_change_time": {
        "dtype": "timestamp[ns]",
        "nullable": True,
        "description": "Actual or revised departure time.",
        "ml_role": "feature",
        "merge_key": False,
    },
    "id": {
        "dtype": "string",
        "nullable": False,
        "description": "Unique identifier for one train stop record.",
        "ml_role": "identifier",
        "merge_key": False,
        "note": "Row-level primary key; do not use as ML feature.",
    },
}

_dict_cols = set(DB_DATA_DICTIONARY.keys())
_config_cols = set(VERIFIED_COLUMNS)
assert _dict_cols == _config_cols, (
    f"Column mismatch!\n  dictionary only: {_dict_cols - _config_cols}\n"
    f"  config only: {_config_cols - _dict_cols}"
)


def fetch_hf_sample(
    dataset_id: str,
    length: int = SAMPLE_LENGTH,
    offset: int = 0,
    timeout: int = REQUEST_TIMEOUT_SEC,
) -> tuple[pd.DataFrame, str]:
    params = {
        "dataset": dataset_id,
        "config": "default",
        "split": "train",
        "offset": offset,
        "length": length,
    }
    response = requests.get(HF_ROWS_API, params=params, timeout=timeout)
    response.raise_for_status()
    payload = response.json()
    rows = [item["row"] for item in payload.get("rows", [])]
    if not rows:
        raise ValueError("API returned zero rows.")
    df = pd.DataFrame(rows)
    source = f"HF Dataset Viewer API (offset={offset}, length={length})"
    return df, source


# Verified fallback if API is down (real July 2024 ICE rows — not invented)
FALLBACK_SAMPLE_ROWS = [
    {
        "station_name": "Braunschweig Hbf",
        "xml_station_name": "Braunschweig Hbf",
        "eva": "08000049",
        "train_number": "292",
        "line_number": None,
        "final_destination_station": "Berlin Ostbahnhof",
        "delay_in_min": 0,
        "time": "2024-07-01T00:01:00",
        "is_canceled": False,
        "train_type": "ICE",
        "train_line_ride_id": "-8314087786935469967",
        "train_line_station_num": 14,
        "arrival_planned_time": "2024-06-30T23:59:00",
        "arrival_change_time": "2024-06-30T23:57:00",
        "departure_planned_time": "2024-07-01T00:01:00",
        "departure_change_time": "2024-07-01T00:01:00",
        "id": "-8314087786935469967-2406301659-14",
    },
    {
        "station_name": "Mainz Hbf",
        "xml_station_name": "Mainz Hbf",
        "eva": "08000240",
        "train_number": "920",
        "line_number": None,
        "final_destination_station": "Hamburg-Altona",
        "delay_in_min": 0,
        "time": "2024-07-01T00:01:00",
        "is_canceled": False,
        "train_type": "ICE",
        "train_line_ride_id": "8503777394879306018",
        "train_line_station_num": 3,
        "arrival_planned_time": "2024-06-30T23:59:00",
        "arrival_change_time": "2024-06-30T23:59:00",
        "departure_planned_time": "2024-07-01T00:01:00",
        "departure_change_time": "2024-07-01T00:01:00",
        "id": "8503777394879306018-2406302324-3",
    },
    {
        "station_name": "München Hbf",
        "xml_station_name": "München Hbf",
        "eva": "08000261",
        "train_number": "618",
        "line_number": None,
        "final_destination_station": "Kiel Hbf",
        "delay_in_min": 1,
        "time": "2024-07-01T00:02:00",
        "is_canceled": False,
        "train_type": "ICE",
        "train_line_ride_id": "2686007473625185344",
        "train_line_station_num": 1,
        "arrival_planned_time": None,
        "arrival_change_time": None,
        "departure_planned_time": "2024-07-01T00:01:00",
        "departure_change_time": "2024-07-01T00:02:00",
        "id": "2686007473625185344-2407010001-1",
    },
    {
        "station_name": "Essen Hbf",
        "xml_station_name": "Essen Hbf",
        "eva": "08000098",
        "train_number": "892",
        "line_number": None,
        "final_destination_station": "Köln Hbf",
        "delay_in_min": 1,
        "time": "2024-07-01T00:03:00",
        "is_canceled": False,
        "train_type": "ICE",
        "train_line_ride_id": "-309830498189753415",
        "train_line_station_num": 13,
        "arrival_planned_time": "2024-07-01T00:00:00",
        "arrival_change_time": "2024-07-01T00:01:00",
        "departure_planned_time": "2024-07-01T00:02:00",
        "departure_change_time": "2024-07-01T00:03:00",
        "id": "-309830498189753415-2406301934-13",
    },
    {
        "station_name": "Hamburg-Altona",
        "xml_station_name": "Hamburg-Altona",
        "eva": "08002553",
        "train_number": "730",
        "line_number": None,
        "final_destination_station": "Hamburg-Altona",
        "delay_in_min": 0,
        "time": "2024-07-01T00:04:00",
        "is_canceled": False,
        "train_type": "ICE",
        "train_line_ride_id": "-3889563661905709281",
        "train_line_station_num": 13,
        "arrival_planned_time": "2024-07-01T00:04:00",
        "arrival_change_time": "2024-07-01T00:04:00",
        "departure_planned_time": None,
        "departure_change_time": None,
        "id": "-3889563661905709281-2406301936-13",
    },
]

try:
    sample_df, sample_source = fetch_hf_sample(HF_DATASET_ID, length=SAMPLE_LENGTH)
except Exception as api_error:
    print(f"WARNING: HF API unavailable ({api_error}). Using verified fallback sample.")
    sample_df = pd.DataFrame(FALLBACK_SAMPLE_ROWS)
    sample_source = "Verified fallback (Hugging Face Dataset Viewer, July 2024)"

missing_cols = [c for c in VERIFIED_COLUMNS if c not in sample_df.columns]
if missing_cols:
    raise ValueError(
        f"Unexpected schema — missing columns: {missing_cols}\n"
        "Re-inspect https://huggingface.co/datasets/piebro/deutsche-bahn-data"
    )

sample_df = sample_df[VERIFIED_COLUMNS]
ice_df = sample_df[sample_df["train_type"] == ICE_FILTER].copy()

timestamp_cols = [
    "time",
    "arrival_planned_time",
    "arrival_change_time",
    "departure_planned_time",
    "departure_change_time",
]
for col in timestamp_cols:
    sample_df[col] = pd.to_datetime(sample_df[col], errors="coerce")
    if not ice_df.empty:
        ice_df[col] = pd.to_datetime(ice_df[col], errors="coerce")


def profile_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "column": df.columns,
            "dtype": [str(df[c].dtype) for c in df.columns],
            "null_count": [int(df[c].isna().sum()) for c in df.columns],
            "null_pct": [round(100 * df[c].isna().mean(), 1) for c in df.columns],
            "n_unique": [int(df[c].nunique(dropna=True)) for c in df.columns],
        }
    ).sort_values("null_pct", ascending=False).reset_index(drop=True)


full_profile = profile_dataframe(sample_df)
ice_profile = profile_dataframe(ice_df) if not ice_df.empty else pd.DataFrame()

train_type_counts = (
    sample_df["train_type"]
    .value_counts(dropna=False)
    .rename_axis("train_type")
    .reset_index(name="count")
)

ice_delay_stats = {}
if not ice_df.empty:
    ice_delay_stats = {
        "min": int(ice_df["delay_in_min"].min()),
        "max": int(ice_df["delay_in_min"].max()),
        "mean": round(float(ice_df["delay_in_min"].mean()), 2),
        "negative_count": int((ice_df["delay_in_min"] < 0).sum()),
        "zero_count": int((ice_df["delay_in_min"] == 0).sum()),
        "positive_count": int((ice_df["delay_in_min"] > 0).sum()),
    }

dictionary_artifact = {
    "metadata": {
        "notebook": config["project"]["notebook"],
        "section": "Section 3",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "hf_dataset_id": HF_DATASET_ID,
        "hf_url": config["data_sources"]["deutsche_bahn"]["hf_url"],
        "sample_source": sample_source,
        "sample_rows_total": int(len(sample_df)),
        "sample_rows_ice": int(len(ice_df)),
        "ice_filter": ICE_FILTER,
        "target_months": TARGET_MONTHS,
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
    },
    "columns": DB_DATA_DICTIONARY,
    "column_order": VERIFIED_COLUMNS,
    "sample_profile": {
        "train_type_distribution": train_type_counts.to_dict(orient="records"),
        "full_null_profile": full_profile.to_dict(orient="records"),
        "ice_null_profile": ice_profile.to_dict(orient="records") if not ice_profile.empty else [],
        "ice_delay_stats": ice_delay_stats,
    },
    "ice_scope_notes": [
        "One row = one train stop at one station.",
        f"Filter: train_type == '{ICE_FILTER}'.",
        f"Primary ML task: regression on {PRIMARY_TARGET}; metric = MAE.",
        f"Target months: {', '.join(TARGET_MONTHS)}.",
        "line_number is frequently null for ICE — expected, not a data error.",
        "departure_planned_time is the primary temporal anchor for weather merge.",
        "No latitude/longitude — geocode eva in Notebook 04.",
    ],
    "columns_not_for_modeling": ["id", "train_line_ride_id"],
    "proposed_merge_keys": {
        "station": "eva → latitude, longitude (after geocoding)",
        "time": "departure_planned_time rounded to hour → Open-Meteo hourly.time",
    },
}

with DB_DICTIONARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(dictionary_artifact, f, indent=2, ensure_ascii=False)

_ = load_db_data_dictionary()

sep = "=" * 72
print(sep)
print("Section 3: Deutsche Bahn Data Dictionary & ICE Scope")
print(sep)
print(f"Dataset        : {HF_DATASET_ID}")
print(f"Sample source  : {sample_source}")
print(f"Sample rows    : {len(sample_df)} (all train types in sample)")
print(f"ICE rows in sample: {len(ice_df)}")
print(f"Regression target : {PRIMARY_TARGET}")
print(f"Primary metric    : MAE")
print(f"Target months     : {', '.join(TARGET_MONTHS)}")
print()
print("Train type counts in this 100-row sample:")
print(train_type_counts.to_string(index=False))
print()
print("Saved:", DB_DICTIONARY_PATH)
print(sep)
print("Next: Section 4 — Open-Meteo weather API structure")
print(sep)

# Optional display in Jupyter
display(full_profile)
if not ice_df.empty:
    display(ice_df.head())

Section 3: Deutsche Bahn Data Dictionary & ICE Scope
Dataset        : piebro/deutsche-bahn-data
Sample source  : HF Dataset Viewer API (offset=0, length=100)
Sample rows    : 100 (all train types in sample)
ICE rows in sample: 5
Regression target : delay_in_min
Primary metric    : MAE
Target months     : 2024-07, 2024-08, 2024-09

Train type counts in this 100-row sample:
train_type  count
         S     57
        RB     14
        RE     10
       ICE      5
       Bus      4
       MEX      3
       HLB      2
       FEX      2
       STB      1
       SBB      1
       BRB      1

Saved: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\db_data_dictionary.json
Next: Section 4 — Open-Meteo weather API structure


,column,dtype,null_count,null_pct,n_unique
0,departure_change_time,datetime64[ns],22,22.0,6
1,departure_planned_time,datetime64[ns],22,22.0,6
2,arrival_change_time,datetime64[ns],15,15.0,11
3,arrival_planned_time,datetime64[ns],15,15.0,10
4,line_number,object,7,7.0,38
5,station_name,object,4,4.0,57
6,final_destination_station,object,1,1.0,86
7,train_line_ride_id,object,0,0.0,98
8,train_line_station_num,int64,0,0.0,27
9,is_canceled,bool,0,0.0,2


,station_name,xml_station_name,eva,train_number,line_number,final_destination_station,delay_in_min,time,is_canceled,train_type,train_line_ride_id,train_line_station_num,arrival_planned_time,arrival_change_time,departure_planned_time,departure_change_time,id
17,Braunschweig Hbf,Braunschweig Hbf,08000049,292,None,Berlin Ostbahnhof,0,2024-07-01 00:01:00,False,ICE,-8314087786935469967,14,2024-06-30 23:59:00,2024-06-30 23:57:00,2024-07-01 00:01:00,2024-07-01 00:01:00,-8314087786935469967-2406301659-14
23,Mainz Hbf,Mainz Hbf,08000240,920,None,Hamburg-Altona,0,2024-07-01 00:01:00,False,ICE,8503777394879306018,3,2024-06-30 23:59:00,2024-06-30 23:59:00,2024-07-01 00:01:00,2024-07-01 00:01:00,8503777394879306018-2406302324-3
32,München Hbf,München Hbf,08000261,618,None,Kiel Hbf,1,2024-07-01 00:02:00,False,ICE,2686007473625185344,1,NaT,NaT,2024-07-01 00:01:00,2024-07-01 00:02:00,2686007473625185344-2407010001-1
65,Essen Hbf,Essen Hbf,08000098,892,None,Köln Hbf,1,2024-07-01 00:03:00,False,ICE,-309830498189753415,13,2024-07-01 00:00:00,2024-07-01 00:01:00,2024-07-01 00:02:00,2024-07-01 00:03:00,-309830498189753415-2406301934-13
83,Hamburg-Altona,Hamburg-Altona,08002553,730,None,Hamburg-Altona,0,2024-07-01 00:04:00,False,ICE,-3889563661905709281,13,2024-07-01 00:04:00,2024-07-01 00:04:00,NaT,NaT,-3889563661905709281-2406301936-13


# Section 4: Open-Meteo Historical Weather API

**Why this section**  
Weather features will help predict `delay_in_min` (regression, MAE). We must understand the real API before merging.

**What we do**  
1. Document the 8 hourly weather variables we will use.  
2. Call Open-Meteo Archive with a **demo** Berlin location for 1 day.  
3. Save dictionary + cached JSON response (so we don’t re-call the API for inspection).

**Important**  
This demo uses fixed Berlin coordinates. Notebook 04 geocodes each `eva` station and fetches weather for **2024-07, 2024-08, 2024-09**.

**Outputs**  
- `data/reference/weather_data_dictionary.json`  
- `data/reference/open_meteo_sample_response.json`

In [4]:
# =============================================================================
# Notebook 01 | Section 4: Open-Meteo Historical Weather API
# =============================================================================
# Self-contained: loads project_config + db_data_dictionary from disk.
# Calls real Open-Meteo Archive API (demo only).
# Saves weather_data_dictionary.json + open_meteo_sample_response.json
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import requests


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError(
        "Cannot find data/reference/project_config.json. Run Section 1 first."
    )


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"
DB_DICTIONARY_PATH = REFERENCE_DIR / "db_data_dictionary.json"
WEATHER_DICTIONARY_PATH = REFERENCE_DIR / "weather_data_dictionary.json"
SAMPLE_RESPONSE_PATH = REFERENCE_DIR / "open_meteo_sample_response.json"

OPEN_METEO_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
OPEN_METEO_DOCS_URL = "https://open-meteo.com/en/docs/historical-weather-api"
REQUEST_TIMEOUT_SEC = 60

# Demo only — Notebook 04 uses per-station geocoded coordinates
DEMO_LATITUDE = 52.525
DEMO_LONGITUDE = 13.369
DEMO_START_DATE = "2024-07-01"
DEMO_END_DATE = "2024-07-01"
DEMO_TIMEZONE = "Europe/Berlin"

HOURLY_VARIABLES = [
    "temperature_2m",
    "precipitation",
    "rain",
    "snowfall",
    "windspeed_10m",
    "windgusts_10m",
    "weather_code",
    "visibility",
]


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


config = load_json(CONFIG_PATH)
db_dictionary = load_json(DB_DICTIONARY_PATH)
TARGET_MONTHS = config["scope"]["target_months"]
PRIMARY_TARGET = config["ml_tasks"]["primary"]["target"]

WEATHER_VARIABLE_DOCS: dict[str, dict[str, Any]] = {
    "temperature_2m": {
        "unit": "°C",
        "valid_time": "instant",
        "description": "Air temperature at 2 metres above ground.",
        "relevance": "May relate to delay magnitude (infrastructure / operations).",
    },
    "precipitation": {
        "unit": "mm",
        "valid_time": "preceding_hour_sum",
        "description": "Total precipitation in the preceding hour.",
        "relevance": "Wet conditions may increase expected delay minutes.",
    },
    "rain": {
        "unit": "mm",
        "valid_time": "preceding_hour_sum",
        "description": "Liquid rain in the preceding hour.",
        "relevance": "Often redundant with precipitation — check in EDA/features.",
    },
    "snowfall": {
        "unit": "cm",
        "valid_time": "preceding_hour_sum",
        "description": "Snowfall in the preceding hour.",
        "relevance": "May be near-zero in summer months; still requested for completeness.",
    },
    "windspeed_10m": {
        "unit": "km/h",
        "valid_time": "instant",
        "description": "Wind speed at 10 metres.",
        "relevance": "Strong wind may affect high-speed running.",
    },
    "windgusts_10m": {
        "unit": "km/h",
        "valid_time": "instant",
        "description": "Maximum wind gusts at 10 metres.",
        "relevance": "Short severe wind events.",
    },
    "weather_code": {
        "unit": "wmo_code",
        "valid_time": "instant",
        "description": "WMO weather interpretation code (categorical).",
        "relevance": "Encodes fog, thunderstorm, snow in one field.",
    },
    "visibility": {
        "unit": "m",
        "valid_time": "instant",
        "description": "Horizontal visibility distance.",
        "relevance": "Low visibility can cause speed restrictions.",
    },
}


def fetch_open_meteo_archive(
    latitude: float,
    longitude: float,
    start_date: str,
    end_date: str,
    hourly_variables: list[str],
    timezone: str = "Europe/Berlin",
    timeout: int = REQUEST_TIMEOUT_SEC,
) -> dict:
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(hourly_variables),
        "timezone": timezone,
    }
    response = requests.get(OPEN_METEO_ARCHIVE_URL, params=params, timeout=timeout)
    response.raise_for_status()
    return response.json()


def parse_hourly_response(api_response: dict) -> pd.DataFrame:
    if "hourly" not in api_response:
        raise ValueError("API response missing 'hourly' key.")
    hourly = api_response["hourly"]
    if "time" not in hourly:
        raise ValueError("API response missing 'hourly.time'.")
    df = pd.DataFrame(hourly)
    df["time"] = pd.to_datetime(df["time"])
    df["latitude"] = api_response.get("latitude")
    df["longitude"] = api_response.get("longitude")
    df["timezone"] = api_response.get("timezone")
    return df


print("Calling Open-Meteo Archive API (demo)...")
api_response = fetch_open_meteo_archive(
    latitude=DEMO_LATITUDE,
    longitude=DEMO_LONGITUDE,
    start_date=DEMO_START_DATE,
    end_date=DEMO_END_DATE,
    hourly_variables=HOURLY_VARIABLES,
    timezone=DEMO_TIMEZONE,
)

with SAMPLE_RESPONSE_PATH.open("w", encoding="utf-8") as f:
    json.dump(api_response, f, indent=2)

weather_df = parse_hourly_response(api_response)

response_hourly_keys = set(api_response["hourly"].keys())
expected_keys = set(HOURLY_VARIABLES) | {"time"}
missing_vars = expected_keys - response_hourly_keys
if missing_vars:
    raise ValueError(f"API response missing hourly variables: {missing_vars}")

weather_dictionary = {
    "metadata": {
        "notebook": config["project"]["notebook"],
        "section": "Section 4",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "api_url": OPEN_METEO_ARCHIVE_URL,
        "docs_url": OPEN_METEO_DOCS_URL,
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
        "target_months": TARGET_MONTHS,
        "demo_request": {
            "latitude": DEMO_LATITUDE,
            "longitude": DEMO_LONGITUDE,
            "start_date": DEMO_START_DATE,
            "end_date": DEMO_END_DATE,
            "timezone": DEMO_TIMEZONE,
            "hourly_variables": HOURLY_VARIABLES,
            "note": "Demo only — Notebook 04 uses per-station geocoded coordinates for all target months.",
        },
        "sample_response_path": str(SAMPLE_RESPONSE_PATH),
        "sample_rows": int(len(weather_df)),
    },
    "required_request_parameters": {
        "latitude": "WGS84 latitude — obtained by geocoding eva in Notebook 04",
        "longitude": "WGS84 longitude — obtained by geocoding eva in Notebook 04",
        "start_date": "YYYY-MM-DD — match railway month being processed",
        "end_date": "YYYY-MM-DD — match railway month being processed",
        "hourly": "Comma-separated list of weather variables",
        "timezone": "Europe/Berlin — aligns with German rail timestamps",
    },
    "response_top_level_keys": {
        "latitude": "Grid-snapped latitude returned by API",
        "longitude": "Grid-snapped longitude returned by API",
        "timezone": "Timezone used for hourly.time",
        "hourly_units": "Unit for each hourly variable",
        "hourly": "Object containing parallel time-series arrays",
    },
    "hourly_variables": WEATHER_VARIABLE_DOCS,
    "hourly_time_format": {
        "example": str(weather_df["time"].iloc[0]),
        "pattern": "YYYY-MM-DDTHH:00 in requested timezone",
        "merge_rule": (
            "Floor railway departure_planned_time to hour, "
            "then join on (eva/station coords + hour)"
        ),
    },
    "merge_plan": {
        "railway_keys": db_dictionary["proposed_merge_keys"],
        "weather_keys": {
            "location": "latitude + longitude (after geocoding eva)",
            "time": "hourly.time (hour-rounded)",
        },
        "join_type": "left join (keep all ICE railway rows; weather may be null if missing)",
        "implemented_in": "Notebook 04",
        "months": TARGET_MONTHS,
    },
    "verified_response_units": api_response.get("hourly_units", {}),
    "columns_not_in_weather_api": [
        "station_name",
        "eva",
        "train_number",
        "delay_in_min",
        "note: weather API is location+time only — no railway fields",
    ],
}

with WEATHER_DICTIONARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(weather_dictionary, f, indent=2, ensure_ascii=False)

loaded_weather_dict = load_json(WEATHER_DICTIONARY_PATH)

sep = "=" * 72
print(sep)
print("Section 4: Open-Meteo Historical Weather API")
print(sep)
print(f"API URL         : {OPEN_METEO_ARCHIVE_URL}")
print(f"Demo coords     : ({DEMO_LATITUDE}, {DEMO_LONGITUDE}) — Berlin area")
print(f"Demo date       : {DEMO_START_DATE}")
print(f"Timezone        : {DEMO_TIMEZONE}")
print(f"Regression target: {PRIMARY_TARGET} | metric: MAE")
print(f"Target months   : {', '.join(TARGET_MONTHS)}")
print()
print("--- API RESPONSE METADATA ---")
print(f"  Returned latitude  : {api_response.get('latitude')}")
print(f"  Returned longitude : {api_response.get('longitude')}")
print(f"  Timezone           : {api_response.get('timezone')}")
print()
print("--- HOURLY UNITS ---")
for var, unit in api_response.get("hourly_units", {}).items():
    print(f"  {var}: {unit}")
print()
print(f"--- PARSED DATAFRAME ({len(weather_df)} hourly rows) ---")
display_cols = ["time"] + HOURLY_VARIABLES
print(weather_df[display_cols].head(3).to_string(index=False))
print()
print("--- SAVED ---")
print(f"  {WEATHER_DICTIONARY_PATH}")
print(f"  {SAMPLE_RESPONSE_PATH}")
print(f"  Reload OK: {len(loaded_weather_dict['hourly_variables'])} variables")
print(sep)
print("Next: Section 5 — merge strategy & pipeline design")
print(sep)

Calling Open-Meteo Archive API (demo)...
Section 4: Open-Meteo Historical Weather API
API URL         : https://archive-api.open-meteo.com/v1/archive
Demo coords     : (52.525, 13.369) — Berlin area
Demo date       : 2024-07-01
Timezone        : Europe/Berlin
Regression target: delay_in_min | metric: MAE
Target months   : 2024-07, 2024-08, 2024-09

--- API RESPONSE METADATA ---
  Returned latitude  : 52.54833
  Returned longitude : 13.407822
  Timezone           : Europe/Berlin

--- HOURLY UNITS ---
  time: iso8601
  temperature_2m: °C
  precipitation: mm
  rain: mm
  snowfall: cm
  windspeed_10m: km/h
  windgusts_10m: km/h
  weather_code: wmo code
  visibility: undefined

--- PARSED DATAFRAME (24 hourly rows) ---
               time  temperature_2m  precipitation  rain  snowfall  windspeed_10m  windgusts_10m  weather_code visibility
2024-07-01 00:00:00            17.0            0.0   0.0       0.0           12.6           21.6             3       None
2024-07-01 01:00:00            1

# Section 5: Merge Strategy, Pipeline Design & Notebook 01 Summary

**Why this section**  
We write the join blueprint before coding Notebook 04, so merge keys and leakage rules are fixed.

**Join idea (simple)**  
- Station: `eva` → geocode → lat/lon → weather for that station  
- Time: floor `departure_planned_time` to hour → match hourly weather  
- Join type: **left** (keep all ICE rows)

**Project decisions locked here**  
- Regression target: `delay_in_min`  
- Metric: **MAE only**  
- Months: **2024-07, 2024-08, 2024-09**  
- No classification models  

**Output**  
`data/reference/merge_strategy.json`

**Next notebook**  
Notebook 02 — acquire ICE data for all three months.

In [5]:
# =============================================================================
# Notebook 01 | Section 5: Merge Strategy, Pipeline Design & Notebook Summary
# =============================================================================
# Self-contained: loads ALL reference JSON files from disk.
# Saves merge_strategy.json — blueprint for Notebook 04.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError(
        "Cannot find data/reference/project_config.json. Run Section 1 first."
    )


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"

CONFIG_PATH = REFERENCE_DIR / "project_config.json"
RESEARCH_PATH = REFERENCE_DIR / "research_framework.json"
DB_DICT_PATH = REFERENCE_DIR / "db_data_dictionary.json"
WEATHER_DICT_PATH = REFERENCE_DIR / "weather_data_dictionary.json"
SAMPLE_RESPONSE_PATH = REFERENCE_DIR / "open_meteo_sample_response.json"
MERGE_STRATEGY_PATH = REFERENCE_DIR / "merge_strategy.json"


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing: {path}\nRun the earlier section that creates this file first."
        )
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


required_files = {
    "Section 1 — project_config": CONFIG_PATH,
    "Section 2 — research_framework": RESEARCH_PATH,
    "Section 3 — db_data_dictionary": DB_DICT_PATH,
    "Section 4 — weather_data_dictionary": WEATHER_DICT_PATH,
}

print("Checking prior Section artifacts:")
for label, path in required_files.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"  [{status}] {label}: {path.name}")

missing = [label for label, path in required_files.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Complete these sections first: {missing}")

config = load_json(CONFIG_PATH)
research = load_json(RESEARCH_PATH)
db_dict = load_json(DB_DICT_PATH)
weather_dict = load_json(WEATHER_DICT_PATH)

TARGET_MONTHS = config["scope"]["target_months"]
PRIMARY_TARGET = config["ml_tasks"]["primary"]["target"]
PRIMARY_METRIC = config["ml_tasks"]["primary"]["primary_metric"]
AUTHORS = config["project"].get("authors", [])

MERGE_STRATEGY: dict[str, Any] = {
    "metadata": {
        "notebook": config["project"]["notebook"],
        "section": "Section 5",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "implemented_in": "Notebook 04 — Data Integration",
        "depends_on": [str(p) for p in required_files.values()],
        "primary_task": "regression",
        "primary_target": PRIMARY_TARGET,
        "primary_metric": PRIMARY_METRIC,
        "target_months": TARGET_MONTHS,
        "authors": AUTHORS,
    },
    "pipeline_steps": [
        {
            "step": 1,
            "notebook": "02",
            "action": "Load monthly Parquet from Hugging Face for each target month",
            "months": TARGET_MONTHS,
            "input": "hf://datasets/piebro/deutsche-bahn-data/monthly_processed_data/data-YYYY-MM.parquet",
            "output": "data/processed/ice_raw_YYYY-MM.parquet",
        },
        {
            "step": 2,
            "notebook": "02",
            "action": "Filter train_type == 'ICE' (per month)",
            "input": "raw monthly parquet",
            "output": "ICE-only subset saved to disk",
        },
        {
            "step": 3,
            "notebook": "03",
            "action": "Standardize timestamps to Europe/Berlin; create departure_planned_hour_naive",
            "columns": [
                "time",
                "arrival_planned_time",
                "arrival_change_time",
                "departure_planned_time",
                "departure_change_time",
            ],
            "output": "data/processed/ice_cleaned_YYYY-MM.parquet",
        },
        {
            "step": 4,
            "notebook": "04",
            "action": "Geocode unique eva station IDs to latitude/longitude",
            "input": "unique eva values from ICE data (across months)",
            "output": "data/reference/station_coordinates.json",
            "api": "Nominatim / OpenStreetMap (documented in Notebook 04)",
        },
        {
            "step": 5,
            "notebook": "04",
            "action": "Fetch hourly weather per station-month from Open-Meteo Archive",
            "input": "station coordinates + month date range",
            "output": "data/processed/weather_by_station_YYYY-MM.parquet",
            "cache_rule": "Cache per station-month to avoid repeated API calls",
        },
        {
            "step": 6,
            "notebook": "04",
            "action": "Create merge keys on both sides",
            "railway_key": "eva + departure_planned_hour_naive",
            "weather_key": "eva + weather_time (hourly)",
        },
        {
            "step": 7,
            "notebook": "04",
            "action": "Left join weather onto railway rows (per month)",
            "output": "data/processed/ice_weather_merged_YYYY-MM.parquet",
        },
        {
            "step": 8,
            "notebook": "04",
            "action": "Run validation checks; optionally concat months for EDA/ML",
            "output": "data/reference/merge_validation_report_YYYY-MM.json",
        },
    ],
    "merge_keys": {
        "railway_side": {
            "station_key": {
                "source_column": "eva",
                "transform": "lookup eva in station_coordinates.json → latitude, longitude",
                "why": "Railway data has eva but no coordinates; weather API needs lat/lon",
            },
            "time_key": {
                "source_column": "departure_planned_time",
                "transform": "floor to hour in Europe/Berlin timezone",
                "output_column": "departure_planned_hour_naive",
                "why": (
                    "Open-Meteo returns hourly data. "
                    "Planned departure (not actual) avoids target leakage."
                ),
            },
        },
        "weather_side": {
            "station_key": {
                "column": "eva",
                "note": "Weather fetched and stored per eva after geocoding",
            },
            "time_key": {
                "column": "weather_time",
                "format": "YYYY-MM-DDTHH:00 in Europe/Berlin",
                "source": "hourly.time from Open-Meteo response",
            },
        },
        "join_condition": (
            "railway.eva == weather.eva AND "
            "railway.departure_planned_hour_naive == weather.weather_time"
        ),
        "join_type": "left",
        "join_type_rationale": (
            "Keep all ICE railway rows. "
            "Rows without weather match are flagged, not dropped."
        ),
    },
    "merge_key_justification": {
        "eva_not_station_name": (
            "eva is a standardized ID; station_name can be null or inconsistent"
        ),
        "departure_planned_not_actual": (
            "Using arrival_change_time or departure_change_time would leak "
            "the delay outcome into the features"
        ),
        "hour_not_minute": (
            "Weather variables are hourly; minute-level join is not supported cleanly"
        ),
        "left_not_inner": (
            "Inner join would silently drop stops with missing weather"
        ),
    },
    "validation_checks": [
        {
            "check": "merge_rate",
            "rule": "≥ 90% of mergeable ICE rows have non-null weather values",
            "action_if_fail": "Investigate missing geocoding or date range gaps",
        },
        {
            "check": "no_duplicate_row_keys",
            "rule": "id column must be unique after merge",
            "action_if_fail": "Inspect join — weather side may have duplicate hours",
        },
        {
            "check": "timezone_consistency",
            "rule": "All timestamps in Europe/Berlin; no mixed UTC/local",
            "action_if_fail": "Re-run Notebook 03 standardization",
        },
        {
            "check": "no_target_leakage",
            "rule": (
                "Features must not include arrival_change_time, "
                "departure_change_time, or delay_in_min"
            ),
            "action_if_fail": "Remove leaky columns before modeling in Notebook 06",
        },
        {
            "check": "weather_null_pattern",
            "rule": "Null weather should not be systematically missing by station",
            "action_if_fail": "Fix geocoding for affected stations",
        },
        {
            "check": "row_count_preserved",
            "rule": "Row count after left join == row count before join",
            "action_if_fail": "Join created duplicates — inspect weather cache",
        },
    ],
    "expected_merged_columns": {
        "railway_columns": config["db_verified_columns"],
        "weather_columns": list(weather_dict["hourly_variables"].keys()),
        "derived_columns": [
            "departure_planned_hour",
            "departure_planned_hour_naive",
            "latitude",
            "longitude",
            "weather_matched",
            "mergeable",
        ],
    },
    "columns_excluded_from_features": [
        "id",
        "train_line_ride_id",
        "delay_in_min",
        "arrival_change_time",
        "departure_change_time",
        "note: delay_in_min is the regression TARGET (MAE); change times leak the outcome",
    ],
    "modeling_plan": {
        "task": "regression",
        "target": PRIMARY_TARGET,
        "primary_metric": PRIMARY_METRIC,
        "not_in_scope": ["classification models", "RMSE as required metric"],
        "months": TARGET_MONTHS,
    },
    "output_files": {
        "merged_data_pattern": "data/processed/ice_weather_merged_YYYY-MM.parquet",
        "station_coords": "data/reference/station_coordinates.json",
        "validation_report_pattern": "data/reference/merge_validation_report_YYYY-MM.json",
    },
}

with MERGE_STRATEGY_PATH.open("w", encoding="utf-8") as f:
    json.dump(MERGE_STRATEGY, f, indent=2, ensure_ascii=False)

loaded_strategy = load_json(MERGE_STRATEGY_PATH)

NOTEBOOK_01_CHECKLIST = {
    "sections_completed": {
        "Section 1": CONFIG_PATH.exists(),
        "Section 2": RESEARCH_PATH.exists(),
        "Section 3": DB_DICT_PATH.exists(),
        "Section 4": WEATHER_DICT_PATH.exists(),
        "Section 5": MERGE_STRATEGY_PATH.exists(),
    },
    "artifacts_on_disk": {
        "project_config.json": CONFIG_PATH.exists(),
        "research_framework.json": RESEARCH_PATH.exists(),
        "db_data_dictionary.json": DB_DICT_PATH.exists(),
        "weather_data_dictionary.json": WEATHER_DICT_PATH.exists(),
        "open_meteo_sample_response.json": SAMPLE_RESPONSE_PATH.exists(),
        "merge_strategy.json": MERGE_STRATEGY_PATH.exists(),
    },
    "ready_for_notebook_02": all(
        [
            CONFIG_PATH.exists(),
            RESEARCH_PATH.exists(),
            DB_DICT_PATH.exists(),
            WEATHER_DICT_PATH.exists(),
            MERGE_STRATEGY_PATH.exists(),
        ]
    ),
}

sep = "=" * 72
print(sep)
print("Section 5: Merge Strategy, Pipeline Design & Notebook Summary")
print(sep)
print(f"Project : {config['project']['title']}")
print(f"Authors : {', '.join(AUTHORS) if AUTHORS else '(not set)'}")
print(f"Task    : regression on {PRIMARY_TARGET}")
print(f"Metric  : {PRIMARY_METRIC.upper()} only")
print(f"Months  : {', '.join(TARGET_MONTHS)}")
print()

print("--- MERGE KEYS ---")
rk = loaded_strategy["merge_keys"]["railway_side"]
print(f"  Railway station : {rk['station_key']['source_column']} → geocode → lat/lon")
print(f"  Railway time    : {rk['time_key']['source_column']} → floor to hour")
print(f"  Join condition  : {loaded_strategy['merge_keys']['join_condition']}")
print(f"  Join type       : {loaded_strategy['merge_keys']['join_type']}")
print()

print("--- PIPELINE (Notebooks 02–04) ---")
for step in loaded_strategy["pipeline_steps"]:
    print(f"  Step {step['step']} [NB{step['notebook']}]: {step['action']}")
print()

print("--- VALIDATION CHECKS ---")
for check in loaded_strategy["validation_checks"]:
    print(f"  • {check['check']}: {check['rule']}")
print()

print("--- NOTEBOOK 01 ARTIFACTS ---")
for name, ok in NOTEBOOK_01_CHECKLIST["artifacts_on_disk"].items():
    print(f"  [{'OK' if ok else 'MISSING'}] {name}")
print()
print(f"Merge strategy saved : {MERGE_STRATEGY_PATH}")
print(f"Ready for Notebook 02: {NOTEBOOK_01_CHECKLIST['ready_for_notebook_02']}")
print(sep)
print("Notebook 01 complete. Next: Notebook 02 — multi-month ICE acquisition")
print(sep)

Checking prior Section artifacts:
  [OK] Section 1 — project_config: project_config.json
  [OK] Section 2 — research_framework: research_framework.json
  [OK] Section 3 — db_data_dictionary: db_data_dictionary.json
  [OK] Section 4 — weather_data_dictionary: weather_data_dictionary.json
Section 5: Merge Strategy, Pipeline Design & Notebook Summary
Project : ICE Train Delay Prediction Using Railway Operational Data and Historical Weather Data
Authors : Manikanta Engalligi, abhinay-sambherao
Task    : regression on delay_in_min
Metric  : MAE only
Months  : 2024-07, 2024-08, 2024-09

--- MERGE KEYS ---
  Railway station : eva → geocode → lat/lon
  Railway time    : departure_planned_time → floor to hour
  Join condition  : railway.eva == weather.eva AND railway.departure_planned_hour_naive == weather.weather_time
  Join type       : left

--- PIPELINE (Notebooks 02–04) ---
  Step 1 [NB02]: Load monthly Parquet from Hugging Face for each target month
  Step 2 [NB02]: Filter train_type == '